
# Create Self-managed Vector Search Index

In the previous demo, we chunked the raw PDF document pages into small sections, computed the embeddings, and saved it as a Delta Lake table. Our dataset is now ready. 

Next, we'll configure Databricks Vector Search to ingest data from this table.

Vector search index uses a Vector search endpoint to serve the embeddings (you can think about it as your Vector Search API endpoint). <br/>
Multiple Indexes can use the same endpoint. Let's start by creating one.


**Learning Objectives:**

*By the end of this demo, you will be able to;*

* Set up an endpoint for Vector Search.

* Store the embeddings and their metadata using the Vector Search.

* Inspect the Vector Search endpoint and index using the UI. 

* Retrieve documents from the vector store using similarity search.



## Create a "Self-Managed" Vector Search Index

Setting up a Databricks Vector Search index involves a few key steps. First, you need to decide on the method of providing vector embeddings. Databricks supports three options: 

* providing a source Delta table containing text data
* **providing a source Delta table that contains pre-calculated embeddings**
* using the Direct Vector API to create an index on embeddings stored in a Delta table

In this demo, we will go with the second method. 

Next, we will **create a vector search endpoint**. And in the final step, we will **create a vector search index** from a Delta table


### Setup a Vector Search Endpoint

The first step for creating a Vector Search index is to create a compute endpoint. This endpoint serves the vector search index. You can query and update the endpoint using the REST API or the SDK. 

**🚨IMPORTANT: Vector Search endpoints must be created before running the rest of the demo. These are already created for you in Databricks Lab environment.**

In [0]:
%pip install -U -qq databricks-vectorsearch databricks-sdk
dbutils.library.restartPython()

In [0]:
# assign vs search endpoint by username
vs_endpoint_prefix = "vs_endpoint_"
vs_endpoint_name = vs_endpoint_prefix+str("pih_brand")

print(f"Assigned Vector Search endpoint name: {vs_endpoint_name}.")

In [0]:
from databricks.vector_search.client import VectorSearchClient
from databricks.sdk import WorkspaceClient
import databricks.sdk.service.catalog as c

vsc = VectorSearchClient(disable_notice=True)

endpoint_info = vsc.get_endpoint("vs_endpoint_pih_brand")  # returns a dict
#state_info = endpoint_info.get('state', {})

# Print the full structure
print(endpoint_info)

state_info = endpoint_info.get('endpoint_status', {})
status = state_info.get('state', None)
print(f"Endpoint status: {status}")

In [0]:
import time

def wait_for_vs_endpoint_to_be_ready(vsc, endpoint_name: str, timeout_minutes: int = 10):
    """
    Waits until the Vector Search Endpoint status becomes 'ONLINE'.

    Args:
        vsc: VectorSearchClient instance.
        endpoint_name (str): Name of the Vector Search endpoint.
        timeout_minutes (int): Maximum minutes to wait before timing out (default 10 minutes).

    Raises:
        TimeoutError: If the endpoint does not become ONLINE in the specified timeout.
    """
    start_time = time.time()
    timeout_seconds = timeout_minutes * 60

    while True:
        endpoint_info = vsc.get_endpoint(endpoint_name)
        state_info = endpoint_info.get('endpoint_status', {})
        status = state_info.get('state', None)

        if status == "ONLINE":
            print(f"✅ Vector Search endpoint '{endpoint_name}' is ONLINE!")
            break
        
        elapsed = time.time() - start_time
        if elapsed > timeout_seconds:
            raise TimeoutError(f"❌ Timeout: Vector Search endpoint '{endpoint_name}' was not ONLINE after {timeout_minutes} minutes. Last status: {status}")

        print(f"⏳ Endpoint '{endpoint_name}' status is {status}. Waiting 10 seconds...")
        time.sleep(10)


Check the status of the endpoint

In [0]:
from databricks.vector_search.client import VectorSearchClient
from databricks.sdk import WorkspaceClient
import databricks.sdk.service.catalog as c


vsc = VectorSearchClient(disable_notice=True)

vs_endpoint_name = "vs_endpoint_pih_brand"

# check the status of the endpoint
wait_for_vs_endpoint_to_be_ready(vsc, vs_endpoint_name)
print(f"Endpoint named {vs_endpoint_name} is ready.")

View the Endpoint
After the endpoint is created, you can view your endpoint on the Vector Search Endpoints UI. Click on the endpoint name to see all indexes that are served by the endpoint.

### Connect Delta Table with Vector Search Endpoint

After creating the endpoint, we can create the **vector search index**. The vector search index is created from a Delta table and is optimized to provide real-time approximate nearest neighbor searches. The goal of the search is to identify documents that are similar to the query. 

**Vector search indexes appear in and are governed by Unity Catalog.**

In [0]:
def index_exists(vsc, endpoint_name: str, index_name: str) -> bool:
    """
    Checks if a Vector Search index exists on a given endpoint.

    Args:
        vsc: VectorSearchClient instance.
        endpoint_name (str): Name of the Vector Search endpoint.
        index_name (str): Full name of the index to check.

    Returns:
        bool: True if index exists, False otherwise.
    """
    try:
        vsc.get_index(endpoint_name, index_name)
        return True
    except Exception as e:
        # If index is not found, Databricks API throws an error
        if "RESOURCE_DOES_NOT_EXIST" in str(e):
            return False
        else:
            raise e  # rethrow other unexpected errors

In [0]:
import time

def wait_for_index_to_be_ready(vsc, endpoint_name: str, index_name: str, timeout_minutes: int = 15):
    """
    Waits until the Vector Search index is READY.

    Args:
        vsc: VectorSearchClient instance.
        endpoint_name (str): Name of the Vector Search endpoint.
        index_name (str): Full name of the index.
        timeout_minutes (int): Max wait time before timeout.

    Raises:
        TimeoutError: If the index is not READY in time.
    """
    start_time = time.time()
    timeout_seconds = timeout_minutes * 60

    while True:
        try:
            index_obj = vsc.get_index(endpoint_name, index_name)
            index_metadata = index_obj.describe()

            ready = index_metadata.get('status', {}).get('ready', False)
            detailed_state = index_metadata.get('status', {}).get('detailed_state', 'UNKNOWN')

            if ready:
                print(f"✅ Index '{index_name}' is READY! Detailed state: {detailed_state}")
                break
            else:
                print(f"⏳ Index not ready yet. Detailed state: {detailed_state}. Waiting 10 seconds...")

        except Exception as e:
            print(f"⚠️ Error while fetching index info: {e}")

        elapsed = time.time() - start_time
        if elapsed > timeout_seconds:
            raise TimeoutError(f"❌ Timeout after {timeout_minutes} minutes: Index '{index_name}' not ready. Last known state: {detailed_state}")

        time.sleep(10)


In [0]:
%sql
ALTER TABLE solutions_catalog.pih_schema.brand_data_embeddings
SET TBLPROPERTIES (delta.enableChangeDataFeed = true);


In [0]:
%sql

select * from solutions_catalog.pih_schema.brand_data_embeddings

In [0]:
from databricks.vector_search.client import VectorSearchClient
from databricks.sdk import WorkspaceClient
import databricks.sdk.service.catalog as c

# Initialize the client
vsc = VectorSearchClient(disable_notice=True)

# Variables
source_table_fullname = "solutions_catalog.pih_schema.brand_data_embeddings"
vs_index_fullname = "solutions_catalog.pih_schema.row_text_self_managed_vs_index"
vs_endpoint_name = "vs_endpoint_pih_brand"

# Check if index exists
def index_exists(vsc, endpoint_name: str, index_name: str) -> bool:
    try:
        vsc.get_index(endpoint_name, index_name)
        return True
    except Exception as e:
        if "RESOURCE_DOES_NOT_EXIST" in str(e):
            return False
        else:
            raise e

# Create or sync the index
if not index_exists(vsc, vs_endpoint_name, vs_index_fullname):
    print(f"Creating index {vs_index_fullname} on endpoint {vs_endpoint_name}...")
    vsc.create_delta_sync_index(
        endpoint_name=vs_endpoint_name,
        index_name=vs_index_fullname,
        source_table_name=source_table_fullname,
        pipeline_type="TRIGGERED",  # Manual sync
        primary_key="id",
        embedding_dimension=1024,
        embedding_vector_column="embeddings"
    )
else:
    print(f"Index {vs_index_fullname} already exists. Triggering sync...")
    vsc.get_index(vs_endpoint_name, vs_index_fullname).sync()

# Wait for endpoint and index to be ready
# check the status of the endpoint
wait_for_index_to_be_ready(vsc, vs_endpoint_name,vs_index_fullname)
print(f"Endpoint named {vs_endpoint_name} is ready.")

print("🎯 Ready to use the Vector Search Index!")


In [0]:
def check_index_health(vsc, endpoint_name: str, index_name: str):
    """
    Checks and prints the health and readiness status of a Vector Search Index.

    Args:
        vsc: VectorSearchClient instance.
        endpoint_name (str): Name of the Vector Search endpoint.
        index_name (str): Full name of the index (catalog.schema.index).
    """
    try:
        # Fetch index object
        index_obj = vsc.get_index(endpoint_name, index_name)
        # Fetch detailed metadata
        index_metadata = index_obj.describe()

        # Extract key fields
        ready = index_metadata.get('status', {}).get('ready', False)
        detailed_state = index_metadata.get('status', {}).get('detailed_state', 'UNKNOWN')
        indexed_row_count = index_metadata.get('status', {}).get('indexed_row_count', 'UNKNOWN')
        pipeline_id = index_metadata.get('delta_sync_index_spec', {}).get('pipeline_id', None)

        print("📊 Index Health Check Report")
        print("-" * 40)
        print(f"Index Name         : {index_metadata.get('name', 'UNKNOWN')}")
        print(f"Endpoint Name      : {index_metadata.get('endpoint_name', 'UNKNOWN')}")
        print(f"Index Type         : {index_metadata.get('index_type', 'UNKNOWN')}")
        print(f"Ready              : {ready}")
        print(f"Detailed State     : {detailed_state}")
        print(f"Indexed Row Count  : {indexed_row_count}")
        if pipeline_id:
            print(f"Pipeline ID        : {pipeline_id}")
        else:
            print("Pipeline ID        : ❌ Not available (maybe still initializing)")

        print("-" * 40)

    except Exception as e:
        print(f"⚠️ Error during index health check: {e}")



In [0]:
from databricks.vector_search.client import VectorSearchClient

vsc = VectorSearchClient(disable_notice=True)

# Now call the health checker
check_index_health(vsc, vs_endpoint_name, vs_index_fullname)

## Search for Similar Content

That's all we have to do. Databricks will automatically capture and synchronize new entries in your Delta Lake Table.

Note that depending on your dataset size and model size, index creation can take a few seconds to start and index your embeddings.

Let's give it a try and search for similar content.

**📌 Note:** `similarity_search` also supports a filter parameter. This is useful to add a security layer to your RAG system: you can filter out some sensitive content based on who is doing the call (for example filter on a specific department based on the user preference).

In [0]:
import mlflow.deployments

deploy_client = mlflow.deployments.get_deploy_client("databricks")
question = "show me records KPI is Allstate is a good fit for ABC"
response = deploy_client.predict(endpoint="databricks-gte-large-en", inputs={"input": [question]})
embeddings = [e["embedding"] for e in response.data]
print(embeddings)

In [0]:
# Define your Vector Search details
vs_endpoint_name = "vs_endpoint_pih_brand"
vs_index_fullname = "solutions_catalog.pih_schema.row_text_self_managed_vs_index"

# Perform similarity search using the first embedding vector
results = vsc.get_index(vs_endpoint_name, vs_index_fullname).similarity_search(
    query_vector=embeddings[0],
    columns=["study_id", "kpi_metric_name", "lift_pct", "metric_display_name", "mapping"],
    num_results=10
)

# Format result to align with reranker or downstream use
passages = []
for doc in results.get("result", {}).get("data_array", []):
    new_doc = {
        "study_id": doc[0],
        "kpi_metric_name": doc[1],
        "lift_pct": doc[2],
        "metric_display_name": doc[3],
        "mapping": doc[4]
    }
    passages.append(new_doc)

# Print the most similar records
print(passages)
